# SignalWatch — Marketing Anomaly Detection Agent

Cell-by-cell build: extract daily KPIs from BigQuery -> compute rolling and day-of-week baselines -> flag anomalies.

Run cells top to bottom. Re-run the baseline cells freely while you tune thresholds — they don't re-query BigQuery.

## 1. Install dependencies

In [ ]:
%pip install google-cloud-bigquery pandas numpy db-dtypes google-genai

## 2. Imports and config

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
print(sys.executable)  # sanity check: confirm this is the kernel you expect

import pandas as pd

sys.path.append("src")   # kpi_baseline.py lives in src/, not next to this notebook
import kpi_baseline as kb

PROJECT_ID = ""    # <-- replace with your own GCP project id
SQL_PATH = "sql/01_daily_kpis.sql"    # <-- path to the SQL file, relative to this notebook's location

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 3. Pull daily KPIs from BigQuery

In [ ]:
df = kb.run_query(SQL_PATH, PROJECT_ID)
df["event_date"] = pd.to_datetime(df["event_date"])
df.head()

## 3b. Visualize engagement_rate over time

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["event_date"], df["engagement_rate"], marker="o", markersize=3)
ax.axvline(pd.Timestamp("2020-12-29"), color="red", linestyle="--", label="Dec 29 (suspected level shift)")
ax.set_title("Daily engagement_rate over time")
ax.set_ylabel("engagement_rate")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Build baselines step by step

Broken into separate cells (rather than one `build_baseline_table` call) so you can inspect each stage — useful while you're tuning the day-of-week logic or checking for NaNs in the warm-up period.

In [ ]:
df = kb.add_rolling_baselines(df)
df[["event_date", "conversion_rate", "conversion_rate_avg_7d", "conversion_rate_pct_vs_7d"]].tail(10)

In [ ]:
# engagement_rate had a step-change on 2020-12-29 (confirmed by the chart above) —
# almost certainly a tracking/definition change, not real anomalous behavior.
# Reset its baseline to only look at data from that date forward.
KNOWN_LEVEL_SHIFTS = {"engagement_rate": "2020-12-29"}

df = kb.add_day_of_week_baseline(df, window=3, reset_dates=KNOWN_LEVEL_SHIFTS)
df[["event_date", "day_of_week", "conversion_rate", "conversion_rate_dow_median", "conversion_rate_robust_z"]].tail(10)

In [ ]:
df = kb.flag_anomalies(df)
anomalies = df[df["is_anomaly"]]
anomalies[["event_date", "day_of_week", "anomaly_driver_kpi", "anomaly_driver_z"]]

## 5. Try a different threshold

No need to re-query BigQuery — just re-run `flag_anomalies` on the existing `df`.

In [ ]:
df_sensitive = kb.flag_anomalies(df, threshold=2.0)
df_sensitive[df_sensitive["is_anomaly"]][["event_date", "day_of_week", "anomaly_driver_kpi", "anomaly_driver_z"]]

## 6. Save the baseline table

In [ ]:
df.to_csv("data/daily_kpi_baseline.csv", index=False)
print(f"Saved {len(df)} rows, {int(df['is_anomaly'].sum())} flagged as anomalies.")

## 7. Diagnose an anomaly

Pull the segment breakdown (date x device x medium x source) and, for one flagged day, rank segments by their **contribution** to the overall change — not just by raw size. This is what turns a "conversion dropped 18%" alert into "73% of the decline came from mobile non-brand traffic."

In [ ]:
sys.path.append("src")
import diagnosis as dg

SEGMENT_SQL_PATH = "sql/02_segment_breakdown.sql"

segment_df = dg.run_query(SEGMENT_SQL_PATH, PROJECT_ID)
segment_df["event_date"] = pd.to_datetime(segment_df["event_date"])
segment_df.head()

In [ ]:
# Pick one flagged day to diagnose. Swap this for any date from your
# anomaly diagnostic table above (e.g. a Black Friday / Cyber Monday date).
anomaly_row = df[df["is_anomaly"]].iloc[0]
ANOMALY_DATE = anomaly_row["event_date"]
ANOMALY_KPI = anomaly_row["anomaly_driver_kpi"]

print(f"Diagnosing: {ANOMALY_KPI} on {ANOMALY_DATE.date()}")

ranked = dg.diagnose(segment_df, kpi=ANOMALY_KPI, anomaly_date=ANOMALY_DATE)
ranked

### 7a. Condensed summary — one line per flagged day

Just the top driver for each anomaly. Use this to scan all flagged days at a glance; use the full loop below (7b) only when you want to drill into a specific day's full 5-segment breakdown.

In [ ]:
summary_rows = []
for _, anomaly_row in df[df["is_anomaly"]].iterrows():
    date = anomaly_row["event_date"]
    kpi = anomaly_row["anomaly_driver_kpi"]
    try:
        ranked = dg.diagnose(segment_df, kpi=kpi, anomaly_date=date, top_n=1)
        top = ranked.iloc[0]
        summary_rows.append({
            "event_date": date.date(),
            "kpi": kpi,
            "top_driver": f"{top['device_category']} / {top['medium']}",
            "contribution": round(top["contribution"], 4),
        })
    except ValueError as e:
        summary_rows.append({
            "event_date": date.date(), "kpi": kpi,
            "top_driver": f"(skipped: {e})", "contribution": None,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

### 7b. Full detail for every flagged day

The complete 5-segment breakdown per day — long output, mainly useful when you want to scroll through everything rather than picking one date.

In [ ]:
# Diagnose every flagged anomaly, not just the first one.
# engagement_rate anomalies are skipped with a clear message — the segment
# SQL doesn't track engagement at the segment level yet.

for _, anomaly_row in df[df["is_anomaly"]].iterrows():
    date = anomaly_row["event_date"]
    kpi = anomaly_row["anomaly_driver_kpi"]
    print("=" * 70)
    try:
        ranked = dg.diagnose(segment_df, kpi=kpi, anomaly_date=date)
        print(dg.explain(ranked, kpi=kpi, anomaly_date=date))
    except ValueError as e:
        print(f"Skipped {kpi} on {date.date()}: {e}")

## 8. Generate a natural-language alert

The final step: Monitor -> Detect -> Diagnose -> **Explain/Recommend**.

The LLM only ever sees numbers already computed by the statistical layer — it never detects the anomaly itself or invents figures. Severity is also computed in plain Python (`severity_from_z`), not left to the model, so it's reproducible.

In [ ]:
import llm_alert as la
import os

# Get a free API key from https://aistudio.google.com/apikey, then either:
#   1. Paste it directly below (fine for local experimentation), or
#   2. Set it as an environment variable before starting Jupyter:
#        export GEMINI_API_KEY="your-key-here"
#      and leave the line below as os.environ.get("GEMINI_API_KEY")

GEMINI_API_KEY = ""  # or paste your key as a string here
gemini_client = la.get_client(api_key=GEMINI_API_KEY)

In [ ]:
# Pick one flagged day to turn into an alert. Swap this for any date from
# the summary_df table above.
alert_row = df[df["is_anomaly"]].iloc[0]
alert_date = alert_row["event_date"]
alert_kpi = alert_row["anomaly_driver_kpi"]

current_value = alert_row[alert_kpi]
baseline_value = alert_row[f"{alert_kpi}_dow_median"]
pct_change = (current_value - baseline_value) / baseline_value
severity = la.severity_from_z(alert_row[f"{alert_kpi}_robust_z"])

ranked = dg.diagnose(segment_df, kpi=alert_kpi, anomaly_date=alert_date)

alert_text = la.generate_alert(
    gemini_client,
    kpi=alert_kpi,
    anomaly_date=alert_date,
    current_value=current_value,
    baseline_value=baseline_value,
    pct_change=pct_change,
    severity=severity,
    ranked_segments=ranked,
)

print(alert_text)

## 9. Generate alerts for every flagged day

Same as above, but looped across flagged anomalies. `MAX_ALERTS` below is set to 4 by default — enough to see the pattern and confirm everything works without burning through free-tier quota. Once you're confident, raise it (or set it to `None`) to run all ~43. `engagement_rate` days are skipped, same as in section 7, since the segment SQL doesn't support diagnosing them yet.

In [ ]:
import time

PAUSE_BETWEEN_CALLS_SECONDS = 5  # stay comfortably under free-tier rate limits
MAX_ALERTS = 4  # start small to confirm everything works; raise this (or set to None) once you're confident

all_alerts = []
flagged = df[df["is_anomaly"]].reset_index(drop=True)
if MAX_ALERTS is not None:
    flagged = flagged.head(MAX_ALERTS)

for i, alert_row in flagged.iterrows():
    date = alert_row["event_date"]
    kpi = alert_row["anomaly_driver_kpi"]
    print(f"[{i+1}/{len(flagged)}] {kpi} on {date.date()}...")

    try:
        ranked = dg.diagnose(segment_df, kpi=kpi, anomaly_date=date)
    except ValueError as e:
        print(f"  skipped (diagnosis unavailable): {e}")
        continue

    current_value = alert_row[kpi]
    baseline_value = alert_row[f"{kpi}_dow_median"]
    pct_change = (current_value - baseline_value) / baseline_value
    severity = la.severity_from_z(alert_row[f"{kpi}_robust_z"])

    try:
        alert_text = la.generate_alert(
            gemini_client,
            kpi=kpi,
            anomaly_date=date,
            current_value=current_value,
            baseline_value=baseline_value,
            pct_change=pct_change,
            severity=severity,
            ranked_segments=ranked,
        )
        all_alerts.append({
            "event_date": date.date(), "kpi": kpi, "severity": severity,
            "pct_change": pct_change, "current_value": current_value,
            "baseline_value": baseline_value, "alert": alert_text,
        })
    except Exception as e:
        print(f"  failed after retries: {e}")
        all_alerts.append({
            "event_date": date.date(), "kpi": kpi, "severity": severity,
            "pct_change": pct_change, "current_value": current_value,
            "baseline_value": baseline_value, "alert": f"(generation failed: {e})",
        })

    time.sleep(PAUSE_BETWEEN_CALLS_SECONDS)

print(f"\nDone. Generated {len(all_alerts)} alerts.")

In [ ]:
for a in all_alerts:
    print("=" * 70)
    print(a["alert"])

In [ ]:
# Save all alerts to a file for reference
with open("data/alerts.md", "w") as f:
    for a in all_alerts:
        f.write(f"---\n\n{a['alert']}\n\n")
print("Saved data/alerts.md")

## 10. Generate an HTML report

A color-coded, browser-viewable report — one card per alert, sorted by date, severity shown by color instead of scrolling through plain text.

In [ ]:
import report as rpt

rpt.build_html_report(all_alerts, "data/alerts_report.html")
print("Saved data/alerts_report.html — open this file directly in your browser.")